In [4]:
import warnings
import pandas as pd
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

from xgboost import XGBClassifier


# 경고 숨김부
warnings.filterwarnings("ignore")

# 결과 출력 형식 설정부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 파일 경로 설정부
train_valid_file_path = r"C:\Users\Dell3571\Desktop\vscode\study\SKAX\ott-churn-prediction\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_최종파생변수(80개).csv"
test_file_path = r"C:\Users\Dell3571\Desktop\vscode\study\SKAX\ott-churn-prediction\ott-churn-prediction\kim.kwangil\data\03_1차 파생\260602_테스트용(80개).csv"

# valid AUC에서 (train-valid gap)의 이 비율만큼 페널티를 줌
overfit_penalty = 0.5

# 모델 입력 제외 콜럼 설정부
exclude_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "payment_device",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]


# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()
    mapped = lowered.map(
        {"1": 1, "0": 0, "y": 1, "n": 0, "yes": 1, "no": 0, "true": 1, "false": 0}
    )
    numeric = pd.to_numeric(series, errors="coerce")
    return mapped.where(mapped.notna(), numeric)


# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# 타겟 분포 출력 함수부
def print_target_distribution(y):
    target_dist = (
        pd.DataFrame({"count": y.value_counts().sort_index()})
        .assign(percent=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    )
    print(target_dist)


# 전처리 파이프라인 생성 함수부
def make_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])
    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])


# 데이터 전처리 함수부
def prepare_data(df, selected_features, numeric_features):
    df = df.copy()
    for col in numeric_features:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["is_repurchase_num"] = to_binary(df["is_repurchase"])
    df = df[df["is_repurchase_num"].isin([0, 1])].copy()
    X = df[selected_features].copy()
    y = (df["is_repurchase_num"] == 0).astype(int)
    return X, y


# 데이터 로드부
train_valid_df = pd.read_csv(train_valid_file_path).copy()
test_df = pd.read_csv(test_file_path).copy()

# 전체 콜럼 기준 입력 변수 생성부
selected_features = [col for col in train_valid_df.columns if col not in exclude_cols]

# 범주형 변수 설정부
categorical_features = ["age_group"]

# 숫자형 변수 설정부
numeric_features = [col for col in selected_features if col not in categorical_features]

# 입력 변수, 타겟 변수 생성부
X_train_valid, y_train_valid = prepare_data(train_valid_df, selected_features, numeric_features)
X_test, y_test = prepare_data(test_df, selected_features, numeric_features)

print("분석 기준: 최종 파생 변수 + XGBoost Optuna 5-Fold 튜닝 (과적합 페널티 + Early Stopping)")
print("양성 클래스 기준: is_repurchase == 0")
print(f"train/valid 데이터 수: {len(X_train_valid)}")
print(f"test 데이터 수: {len(X_test)}")
print(f"전체 데이터 수: {len(X_train_valid) + len(X_test)}")
print(f"train/valid 비율: {len(X_train_valid) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"test 비율: {len(X_test) / (len(X_train_valid) + len(X_test)) * 100:.2f}%")
print(f"train/valid 파일 전체 콜럼 수: {len(train_valid_df.columns)}")
print(f"test 파일 전체 콜럼 수: {len(test_df.columns)}")
print(f"모델 입력 제외 콜럼 수: {len(exclude_cols)}")
print(f"사용 변수 수: {len(selected_features)}")
print(f"과적합 페널티 가중치: {overfit_penalty}")
print("사용 변수")
print(selected_features)
print("train/valid 타겟 분포")
print_target_distribution(y_train_valid)
print("test 타겟 분포")
print_target_distribution(y_test)

if y_train_valid.nunique() < 2 or y_train_valid.value_counts().min() < 5:
    print("학습 불가: train/valid 타겟 클래스가 부족합니다.")
elif y_test.nunique() < 2:
    print("평가 불가: test 타겟 클래스가 부족합니다.")
else:
    # XGBoost 불균형 가중치 계산부
    negative_count = (y_train_valid == 0).sum()
    positive_count = (y_train_valid == 1).sum()
    scale_pos_weight = negative_count / positive_count if positive_count > 0 else 1

    # Optuna 목적 함수부
    # 과적합 방지 전략:
    # 1) early_stopping_rounds=50 으로 과도한 트리 생성을 억제
    # 2) 5-fold 평균 train-valid gap에 overfit_penalty를 곱해 목적함수에서 차감
    def objective(trial):
        params = {
            "n_estimators": 1000,  # early stopping으로 실제 사용 개수 자동 결정
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "scale_pos_weight": scale_pos_weight,
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": 42,
            "n_jobs": -1,
        }

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_train_scores = []
        fold_valid_scores = []
        best_iterations = []

        for train_idx, valid_idx in skf.split(X_train_valid, y_train_valid):
            X_fold_train = X_train_valid.iloc[train_idx]
            X_fold_valid = X_train_valid.iloc[valid_idx]
            y_fold_train = y_train_valid.iloc[train_idx]
            y_fold_valid = y_train_valid.iloc[valid_idx]

            # early stopping을 위해 preprocessor를 직접 적용 (Pipeline은 eval_set 미지원)
            preprocessor = make_preprocessor(numeric_features, categorical_features)
            X_fold_train_proc = preprocessor.fit_transform(X_fold_train)
            X_fold_valid_proc = preprocessor.transform(X_fold_valid)

            model = XGBClassifier(**params)
            model.fit(
                X_fold_train_proc, y_fold_train,
                eval_set=[(X_fold_valid_proc, y_fold_valid)],
                early_stopping_rounds=50,
                verbose=False,
            )

            bi = getattr(model, "best_iteration", None)
            best_iterations.append((bi + 1) if bi is not None else 300)

            train_proba = model.predict_proba(X_fold_train_proc)[:, 1]
            valid_proba = model.predict_proba(X_fold_valid_proc)[:, 1]

            fold_train_scores.append(roc_auc_score(y_fold_train, train_proba))
            fold_valid_scores.append(roc_auc_score(y_fold_valid, valid_proba))

        mean_train = sum(fold_train_scores) / len(fold_train_scores)
        mean_valid = sum(fold_valid_scores) / len(fold_valid_scores)
        mean_gap = mean_train - mean_valid
        avg_best_n_estimators = int(sum(best_iterations) / len(best_iterations))

        trial.set_user_attr("mean_valid_roc_auc", round(mean_valid, 4))
        trial.set_user_attr("mean_auc_gap", round(mean_gap, 4))
        trial.set_user_attr("best_n_estimators", avg_best_n_estimators)

        # 과적합(train >> valid)이면 gap만큼 페널티를 주어 일반화 성능 위주로 탐색
        return mean_valid - overfit_penalty * max(0.0, mean_gap)

    # Optuna 튜닝 수행부
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=200, show_progress_bar=True)

    best_trial = study.best_trial
    best_mean_valid = best_trial.user_attrs.get("mean_valid_roc_auc", round(study.best_value, 4))
    best_mean_gap = best_trial.user_attrs.get("mean_auc_gap", None)
    best_n_estimators = best_trial.user_attrs.get("best_n_estimators", 300)

    print("Optuna best 조정 점수 (valid roc_auc - 과적합 페널티)")
    print(round(study.best_value, 4))
    print("Optuna best 5-fold valid roc_auc (페널티 적용 전)")
    print(best_mean_valid)
    print("Optuna best 5-fold train-valid auc gap")
    print(best_mean_gap)
    print(f"최적 n_estimators (early stopping 기준 평균): {best_n_estimators}")
    print("Optuna best params")
    print(study.best_params)

    # 최적 파라미터 기반 최종 모델 학습부
    best_params = {
        **study.best_params,
        "n_estimators": best_n_estimators,
        "scale_pos_weight": scale_pos_weight,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
    }

    print("최종 학습에 사용한 XGBoost 파라미터")
    print(best_params)

    final_clf = Pipeline(steps=[
        ("preprocessor", make_preprocessor(numeric_features, categorical_features)),
        ("model", XGBClassifier(**best_params)),
    ])
    final_clf.fit(X_train_valid, y_train_valid)

    y_train_valid_pred = final_clf.predict(X_train_valid)
    y_test_pred = final_clf.predict(X_test)

    y_train_valid_proba = final_clf.predict_proba(X_train_valid)[:, 1]
    y_test_proba = final_clf.predict_proba(X_test)[:, 1]

    train_valid_roc_auc = roc_auc_score(y_train_valid, y_train_valid_proba)
    test_roc_auc = roc_auc_score(y_test, y_test_proba)
    auc_gap = train_valid_roc_auc - test_roc_auc

    result = {
        "model": "XGBoost_Optuna_5Fold",
        "f1_score": f1_score(y_test, y_test_pred, zero_division=0),
        "train_valid_roc_auc": train_valid_roc_auc,
        "test_roc_auc": test_roc_auc,
        "auc_gap": auc_gap,
        "is_overfit": auc_gap >= 0.05,
    }

    results_df = (
        pd.DataFrame([result])
        .set_index("model")
        [["f1_score", "train_valid_roc_auc", "test_roc_auc", "auc_gap", "is_overfit"]]
        .round(4)
    )

    print("최종 테스트 성능")
    print(results_df.to_string())


[I 2026-06-04 22:02:58,494] A new study created in memory with name: no-name-62eee73b-8788-4815-9c28-4251e9cf6dda


분석 기준: 최종 파생 변수 + XGBoost Optuna 5-Fold 튜닝 (과적합 페널티 + Early Stopping)
양성 클래스 기준: is_repurchase == 0
train/valid 데이터 수: 23081
test 데이터 수: 2545
전체 데이터 수: 25626
train/valid 비율: 90.07%
test 비율: 9.93%
train/valid 파일 전체 콜럼 수: 91
test 파일 전체 콜럼 수: 91
모델 입력 제외 콜럼 수: 11
사용 변수 수: 80
과적합 페널티 가중치: 0.5
사용 변수
['is_promotion', 'is_churn_prevented', 'is_user_verified', 'is_basic', 'is_standard', 'is_premium', 'age_group', 'is_female', 'is_male', 'payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios', 'reg_is_weekend', 'reg_hour_morning', 'reg_hour_afternoon', 'reg_hour_evening', 'reg_hour_night', 'total_watch_count', 'unique_movie', 'watch_days', 'total_watch_time(min)', 'active_ratio', 'watch_per_day', 'avg_watch_time(min)', 'median_watch_time(min)', 'std_watch_time(min)', 'max_watch_time(min)', 'avg_daily_watch_time(min)', 'max_daily_watch_time(min)', 'max_daily_sessions', 'recency', 'avg_gap_between_watch_days', 'max_inactive_gap_days', 'avg_gap_w1_watch_days', 'avg_gap_w2_watc

  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-06-04 22:03:46,152] Trial 0 finished with value: 0.8262096002361219 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 8, 'min_child_weight': 8.960785365368121, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'gamma': 0.7799726016810132, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 2.1423021757741068}. Best is trial 0 with value: 0.8262096002361219.
[I 2026-06-04 22:04:14,227] Trial 1 finished with value: 0.8222051259847054 and parameters: {'learning_rate': 0.06054365855469246, 'max_depth': 6, 'min_child_weight': 1.0636066512540283, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808, 'reg_alpha': 0.0008111941985431928, 'reg_lambda': 0.0008260808399079611}. Best is trial 0 with value: 0.8262096002361219.
[I 2026-06-04 22:05:09,322] Trial 2 finished with value: 0.8376589653721632 and parameters: {'learning_rate': 0.024878734419814436, 'max_depth': 5, 'min_child_weight': 3.6473162849